# NSE AI/ML Predictor — Testing & Validation Notebook

**Sections:**
1. Environment & DB Inspection
2. Feature Engineering Test (Single Stock + Charts)
3. Dataset Build & Stats (Target: >1.5% next-day move)
4. Model Training (Ensemble Stack + Balanced Meta-Learner)
5. **Optimal Threshold Finding (F1 Maximisation)**
6. Validation Metrics — Side-by-Side (Optimal vs 0.50)
7. Pattern Accuracy Breakdown (at Optimal Threshold)
8. Sentiment Analysis Test
9. Full Ranked Predictions — ALL Stocks, High → Low
10. Result Analysis & Charts

## 1. Environment & DB Inspection

In [ ]:
import os, sys, sqlite3, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from config import STOCK_DATA_DB, CORPORATE_DB
print(f'Stock DB  : {STOCK_DATA_DB}  exists={os.path.exists(STOCK_DATA_DB)}')
print(f'Corp DB   : {CORPORATE_DB}  exists={os.path.exists(CORPORATE_DB)}')

In [ ]:
conn = sqlite3.connect(STOCK_DATA_DB)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'hist_%'")
tables = [r[0] for r in cursor.fetchall()]
print(f'Stock tables : {len(tables)}')
for t in tables[:6]:
    df = pd.read_sql_query(f'SELECT Date FROM {t} ORDER BY rowid', conn)
    if not df.empty:
        print(f'  {t:<32} first={df.iloc[0,0]}  last={df.iloc[-1,0]}  rows={len(df)}')
conn.close()

conn2 = sqlite3.connect(CORPORATE_DB)
try:
    n_ann = pd.read_sql_query('SELECT COUNT(*) as n FROM announcements', conn2).iloc[0,0]
    n_sym = pd.read_sql_query('SELECT COUNT(DISTINCT symbol) as n FROM announcements', conn2).iloc[0,0]
    print(f'Corp announcements: {n_ann:,} rows | {n_sym} symbols')
except Exception as e:
    print(f'Corp DB: {e}')
conn2.close()

## 2. Feature Engineering Test

In [ ]:
from ml_predictor import (calc_rsi, calc_macd, calc_bollinger, calc_atr,
                           detect_breakout, detect_gap_up, detect_accumulation,
                           detect_doji, detect_hammer, detect_bullish_engulfing)

TEST_SYMBOL = 'RELIANCE'   # change as needed
TABLE = f'hist_{TEST_SYMBOL.replace("-","_")}'

conn = sqlite3.connect(STOCK_DATA_DB)
df = pd.read_sql_query(
    f'SELECT Date, OpenPrice, HighPrice, LowPrice, ClosePrice, TotalTradedQuantity FROM {TABLE}', conn)
conn.close()

df['parsed_date'] = pd.to_datetime(df['Date'], format='%d-%b-%Y', errors='coerce')
df = df.dropna(subset=['parsed_date']).sort_values('parsed_date').reset_index(drop=True)
for col in ['OpenPrice','HighPrice','LowPrice','ClosePrice','TotalTradedQuantity']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',',''), errors='coerce')
df = df.dropna()

df['RSI']       = calc_rsi(df['ClosePrice'])
macd            = calc_macd(df['ClosePrice'])
df['MACD']      = macd['macd']; df['Signal'] = macd['signal']; df['MACD_hist'] = macd['histogram']
bb              = calc_bollinger(df['ClosePrice'])
df['BB_width']  = bb['bb_width']; df['BB_pos'] = bb['bb_position']
df['ATR']       = calc_atr(df['HighPrice'], df['LowPrice'], df['ClosePrice'])
df['Breakout']  = detect_breakout(df)
df['GapUp']     = detect_gap_up(df)
df['Accum']     = detect_accumulation(df)
df['Doji']      = detect_doji(df)
df['Hammer']    = detect_hammer(df)
df['Engulf']    = detect_bullish_engulfing(df)

print(f'{TEST_SYMBOL} — last 10 rows with all indicators:')
print(df[['Date','ClosePrice','RSI','MACD_hist','BB_width','BB_pos','ATR',
          'Breakout','GapUp','Accum','Doji','Hammer','Engulf']].tail(10).to_string(index=False))

In [ ]:
plot_df = df.tail(180).copy()
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'{TEST_SYMBOL} — Price + RSI + MACD (last 180 days)', fontsize=13, fontweight='bold')
sma20 = plot_df['ClosePrice'].rolling(20).mean()
std20 = plot_df['ClosePrice'].rolling(20).std()
axes[0].plot(plot_df['parsed_date'], plot_df['ClosePrice'], color='#58a6ff', lw=1.5, label='Close')
axes[0].plot(plot_df['parsed_date'], sma20, 'orange', lw=1, linestyle='--', label='SMA20')
axes[0].fill_between(plot_df['parsed_date'], sma20-2*std20, sma20+2*std20, alpha=0.12, color='orange')
bd = plot_df[plot_df['Breakout']==1]
axes[0].scatter(bd['parsed_date'], bd['ClosePrice'], marker='^', color='#4ade80', s=80, zorder=5, label='Breakout')
axes[0].set_ylabel('Price (INR)'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.15)
axes[1].plot(plot_df['parsed_date'], plot_df['RSI'], color='#a78bfa', lw=1.5)
axes[1].axhline(70, color='#f87171', lw=0.8, linestyle='--', label='Overbought')
axes[1].axhline(30, color='#4ade80', lw=0.8, linestyle='--', label='Oversold')
axes[1].set_ylim(0, 100); axes[1].set_ylabel('RSI(14)'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.15)
bar_c = ['#4ade80' if v>=0 else '#f87171' for v in plot_df['MACD_hist']]
axes[2].bar(plot_df['parsed_date'], plot_df['MACD_hist'], color=bar_c, width=0.8)
axes[2].plot(plot_df['parsed_date'], plot_df['MACD'], color='#58a6ff', lw=1, label='MACD')
axes[2].plot(plot_df['parsed_date'], plot_df['Signal'], color='orange', lw=1, label='Signal')
axes[2].set_ylabel('MACD'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.15)
plt.tight_layout(); plt.show()

## 3. Dataset Build & Stats

> **Target fix**: changed from `>2%` to `>1.5%` next-day move to reduce class imbalance.

In [ ]:
from ml_predictor import build_ml_dataset

# Train = all history beyond 60 days | Val = last 2 months (temporal holdout)
train_df, val_df, latest_df = build_ml_dataset(days_lookback=365, validation_days=60)

print(f'Train rows     : {len(train_df):,}')
print(f'Val rows (2mo) : {len(val_df):,}')
print(f'Latest rows    : {len(latest_df):,}  <- stocks to predict today')

print(f'\nClass balance (train): {train_df["Target"].value_counts().to_dict()}')
print(f'Surge rate (train)   : {train_df["Target"].mean()*100:.1f}%  (target >1.5%)')
print(f'\nClass balance (val)  : {val_df["Target"].value_counts().to_dict()}')
print(f'Surge rate (val)     : {val_df["Target"].mean()*100:.1f}%')

In [ ]:
# Class balance visualised
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, df_part, title in zip(axes, [train_df, val_df], ['Train Set', 'Val Set (2 months)']):
    counts = df_part['Target'].value_counts().sort_index()
    ax.bar(['No Surge (0)', 'Surge (1)'], counts.values,
           color=['#f87171', '#4ade80'], edgecolor='black', alpha=0.85)
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts)*0.01, f'{v:,}\n({v/sum(counts)*100:.1f}%)',
                ha='center', fontsize=9)
    ax.set_title(f'{title} — Class Balance', fontweight='bold')
    ax.set_ylabel('Count'); ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

print('\nPattern trigger rates in train:')
pat_cols = ['Pat_Breakout','Pat_Consol','Pat_GapUp','Pat_Accumulate','Pat_Doji','Pat_Hammer','Pat_Engulfing']
for col in pat_cols:
    if col in train_df.columns:
        rate  = train_df[col].mean() * 100
        surge = train_df[train_df[col]==1]['Target'].mean()*100 if train_df[col].sum()>0 else 0
        print(f'  {col:<22}: trigger={rate:.1f}%  surge_when_triggered={surge:.1f}%')

In [ ]:
from ml_predictor import FEATURE_COLS
sample = train_df[FEATURE_COLS+['Target']].fillna(0).sample(min(5000, len(train_df)), random_state=42)
corr = sample.corr()
plt.figure(figsize=(16, 12))
mask = np.zeros_like(corr, dtype=bool); mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Model Training

> **Recall fix applied**: `LogisticRegression(class_weight='balanced')` as meta-learner forces the stacker to predict more positives.

In [ ]:
from ml_predictor import build_and_train, save_model

if val_df.empty:
    from sklearn.model_selection import train_test_split
    print('No temporal val — using 80/20 split')
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

print(f'Training on {len(train_df):,} rows...')
bundle, val_metrics = build_and_train(train_df, val_df)
save_model(bundle, val_metrics)

opt_thresh = bundle['optimal_threshold']
print(f'\nOptimal threshold : {opt_thresh:.2f}')
print(f'Model saved to    : models/ensemble_bundle.pkl')

In [ ]:
# Feature importances from RF base model
rf_model = dict(bundle['base_models'])['rf']
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(10, 9))
colors_imp = ['#4ade80' if v > importances.median() else '#58a6ff' for v in importances]
importances.plot(kind='barh', color=colors_imp)
plt.title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()

## 5. Optimal Threshold Finding (F1 Maximisation)

> **Threshold fix**: Instead of hardcoding `prob >= 0.5`, we scan all thresholds and pick the one with highest F1 on the validation set.

In [ ]:
from ml_predictor import predict_with_bundle
from sklearn.metrics import f1_score, precision_score, recall_score

val_probs = predict_with_bundle(bundle, val_df)
y_val     = val_df['Target'].values

thresholds = np.arange(0.05, 0.95, 0.01)
f1s   = [f1_score(y_val, (val_probs >= t).astype(int), zero_division=0) for t in thresholds]
precs = [precision_score(y_val, (val_probs >= t).astype(int), zero_division=0) for t in thresholds]
recs  = [recall_score(y_val, (val_probs >= t).astype(int), zero_division=0) for t in thresholds]

opt_idx = int(np.argmax(f1s))
opt_t   = thresholds[opt_idx]
print(f'Optimal threshold : {opt_t:.2f}')
print(f'F1  at optimal    : {f1s[opt_idx]*100:.1f}%')
print(f'Prec at optimal   : {precs[opt_idx]*100:.1f}%')
print(f'Rec  at optimal   : {recs[opt_idx]*100:.1f}%')

In [ ]:
# Precision-Recall-F1 vs threshold curve
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, [p*100 for p in precs], color='#58a6ff', lw=2, label='Precision %')
ax.plot(thresholds, [r*100 for r in recs],  color='#f87171', lw=2, label='Recall %')
ax.plot(thresholds, [f*100 for f in f1s],   color='#4ade80', lw=2.5, label='F1 %')
ax.axvline(opt_t, color='#facc15', lw=2, linestyle='--', label=f'Optimal = {opt_t:.2f}')
ax.axvline(0.50,  color='white',   lw=1, linestyle=':',  label='Default = 0.50')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('%')
ax.set_title('Precision / Recall / F1 vs Threshold (Validation Set)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 6. Validation Metrics — Side-by-Side (Optimal vs 0.50)

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, roc_curve, auc, classification_report)

pred_opt = (val_probs >= opt_t).astype(int)
pred_50  = (val_probs >= 0.50).astype(int)

print(f'Val set: {len(y_val):,} rows | Surge rate: {y_val.mean()*100:.1f}%\n')

rows = []
for label, preds in [(f'Optimal ({opt_t:.2f})', pred_opt), ('Default (0.50)', pred_50)]:
    cm = confusion_matrix(y_val, preds)
    rows.append({
        'Threshold':  label,
        'Accuracy %': round(accuracy_score(y_val, preds)*100, 1),
        'Precision %':round(precision_score(y_val, preds, zero_division=0)*100, 1),
        'Recall %':   round(recall_score(y_val, preds, zero_division=0)*100, 1),
        'F1 %':       round(f1_score(y_val, preds, zero_division=0)*100, 1),
        'TN': cm[0][0], 'FP': cm[0][1],
        'FN': cm[1][0], 'TP': cm[1][1],
        'Predicted Surge': int(preds.sum()),
    })

compare_df = pd.DataFrame(rows).set_index('Threshold')
print(compare_df.to_string())

In [ ]:
# Confusion matrix comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cm_opt = confusion_matrix(y_val, pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Surge','Surge'], yticklabels=['No Surge','Surge'])
axes[0].set_title(f'Optimal Threshold ({opt_t:.2f})', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_50 = confusion_matrix(y_val, pred_50)
sns.heatmap(cm_50, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['No Surge','Surge'], yticklabels=['No Surge','Surge'])
axes[1].set_title('Default Threshold (0.50)', fontweight='bold')
axes[1].set_xlabel('Predicted')

# ROC curve
fpr, tpr, _ = roc_curve(y_val, val_probs)
roc_auc_val = auc(fpr, tpr)
axes[2].plot(fpr, tpr, color='#58a6ff', lw=2, label=f'AUC = {roc_auc_val:.4f}')
axes[2].plot([0,1],[0,1],'k--',lw=0.8)
axes[2].fill_between(fpr, tpr, alpha=0.12, color='#58a6ff')
# Mark operating points
fpr_opt = cm_opt[0][1] / (cm_opt[0][0] + cm_opt[0][1])
tpr_opt = cm_opt[1][1] / (cm_opt[1][0] + cm_opt[1][1])
fpr_50  = cm_50[0][1]  / (cm_50[0][0]  + cm_50[0][1])
tpr_50  = cm_50[1][1]  / (cm_50[1][0]  + cm_50[1][1])
axes[2].scatter([fpr_opt], [tpr_opt], color='#4ade80', s=120, zorder=5, label=f'Optimal {opt_t:.2f}')
axes[2].scatter([fpr_50],  [tpr_50],  color='#f87171', s=120, zorder=5, label='Default 0.50')
axes[2].set_title('ROC Curve', fontweight='bold')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR'); axes[2].legend(); axes[2].grid(alpha=0.2)

plt.tight_layout(); plt.show()

In [ ]:
# Precision-Recall tradeoff at optimal threshold vs 0.50
print('=== Detailed Classification Report at Optimal Threshold ===')
print(classification_report(y_val, pred_opt, target_names=['No Surge', 'Surge']))

# Probability distribution: surges vs non-surges
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(val_probs[y_val==0], bins=50, alpha=0.6, color='#f87171', label='Actual No-Surge', density=True)
ax.hist(val_probs[y_val==1], bins=50, alpha=0.7, color='#4ade80', label='Actual Surge', density=True)
ax.axvline(opt_t, color='#facc15', lw=2, linestyle='--', label=f'Optimal threshold {opt_t:.2f}')
ax.axvline(0.50,  color='white',   lw=1.5, linestyle=':',  label='Default 0.50')
ax.set_title('Predicted Probability Distribution: Surge vs No-Surge', fontweight='bold')
ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 7. Pattern Accuracy Breakdown (at Optimal Threshold)

In [ ]:
from sklearn.metrics import accuracy_score

patterns = {'Breakout':'Pat_Breakout','Consolidation':'Pat_Consol','Gap-Up':'Pat_GapUp',
            'Accumulation':'Pat_Accumulate','Engulfing':'Pat_Engulfing',
            'Hammer':'Pat_Hammer','Doji':'Pat_Doji'}

pat_results = []
for name, col in patterns.items():
    sub = val_df[val_df[col]==1] if col in val_df.columns else pd.DataFrame()
    if len(sub) < 5: continue
    p    = predict_with_bundle(bundle, sub)
    pred = (p >= opt_t).astype(int)
    pat_results.append({
        'Pattern':      name,
        'Count':        len(sub),
        'Accuracy %':   round(accuracy_score(sub['Target'], pred)*100, 1),
        'Precision %':  round(precision_score(sub['Target'], pred, zero_division=0)*100, 1),
        'Recall %':     round(recall_score(sub['Target'], pred, zero_division=0)*100, 1),
        'F1 %':         round(f1_score(sub['Target'], pred, zero_division=0)*100, 1),
        'Surge Rate %': round(sub['Target'].mean()*100, 1),
    })

pat_df = pd.DataFrame(pat_results).sort_values('F1 %', ascending=False)
print(f'=== Pattern Accuracy at Threshold={opt_t:.2f} ===')
print(pat_df.to_string(index=False))

In [ ]:
if not pat_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = range(len(pat_df))
    axes[0].bar([i-0.2 for i in x], pat_df['Precision %'], width=0.2, color='#58a6ff', label='Precision %')
    axes[0].bar([i     for i in x], pat_df['Recall %'],    width=0.2, color='#f87171', label='Recall %')
    axes[0].bar([i+0.2 for i in x], pat_df['F1 %'],        width=0.2, color='#4ade80', label='F1 %')
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(pat_df['Pattern'], rotation=20, ha='right')
    axes[0].set_title(f'Pattern Metrics at Optimal Threshold ({opt_t:.2f})', fontweight='bold')
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.2)

    axes[1].bar(pat_df['Pattern'], pat_df['Surge Rate %'], color='#facc15', alpha=0.85)
    axes[1].axhline(val_df['Target'].mean()*100, color='white', lw=1.5,
                    linestyle='--', label=f'Overall surge rate {val_df["Target"].mean()*100:.1f}%')
    axes[1].set_xticklabels(pat_df['Pattern'], rotation=20, ha='right')
    axes[1].set_title('Actual Surge Rate per Pattern', fontweight='bold')
    axes[1].set_ylabel('%'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.2)

    plt.tight_layout(); plt.show()

## 8. Sentiment Analysis Test

In [ ]:
from ml_predictor import analyze_sentiment, fetch_google_news_sentiment

tests = [
    'Strong profit growth, record revenue, dividend announced',
    'Stock plunges after fraud investigation, CEO resigns',
    'Company wins large government contract, expansion plans',
    'Quarterly loss widens, debt rises, shares decline',
]
print('=== Lexicon Sentiment ===')
for t in tests:
    s = analyze_sentiment(t)
    icon = '🟢' if s > 0.1 else '🔴' if s < -0.1 else '⚪'
    print(f'  {icon} {s:+.3f} | {t}')

print('\n=== Live Google News Sentiment ===')
for sym in ['RELIANCE','TCS','HDFCBANK','INFY','SBIN']:
    r = fetch_google_news_sentiment(sym)
    icon = '🟢' if r['avg_sentiment'] > 0.05 else '🔴' if r['avg_sentiment'] < -0.05 else '⚪'
    print(f'  {sym:<12} {icon} score={r["avg_sentiment"]:+.3f}  articles={r["article_count"]}')
    for h in r['top_headlines'][:2]:
        print(f'    → {h}')

## 9. Full Ranked Predictions — ALL Stocks, High → Low

Predictions use the **optimal threshold** from the bundle.  
`Above_Threshold = YES` → model says **BUY candidate** for tomorrow.

In [ ]:
from ml_predictor import fetch_google_news_sentiment, predict_with_bundle, summarize_patterns, confidence_band

opt_thresh = bundle['optimal_threshold']
total      = len(latest_df)
print(f'Predicting ALL {total} stocks at threshold={opt_thresh:.2f}...')

results = []
for i, (idx, row) in enumerate(latest_df.iterrows(), 1):
    sym  = row['Symbol']
    news = fetch_google_news_sentiment(sym, timeout=5)
    blended = row['AnnSentiment'] * 0.4 + news['avg_sentiment'] * 0.6
    row_copy = row.copy(); row_copy['AnnSentiment'] = blended
    prob = predict_with_bundle(bundle, pd.DataFrame([row_copy]))[0]

    results.append({
        'Symbol':             sym,
        'Latest_Return_%':    round(row['DailyReturn'], 2),
        'RSI':                round(row['RSI'], 1) if pd.notna(row['RSI']) else None,
        'MACD_Histogram':     round(row['MACD_hist'], 4) if pd.notna(row['MACD_hist']) else None,
        'BB_Position':        round(row['BB_position'], 2) if pd.notna(row['BB_position']) else None,
        'ATR_%':              round(row['ATR_pct'], 2) if pd.notna(row['ATR_pct']) else None,
        'Volume_Surge_x':     round(row['VolumeSurge'], 2) if pd.notna(row['VolumeSurge']) else None,
        'Corp_Ann_Sentiment': round(row['AnnSentiment'], 2),
        'News_Sentiment':     round(news['avg_sentiment'], 2),
        'News_Articles':      news['article_count'],
        'Blended_Sentiment':  round(blended, 2),
        'Patterns_Detected':  summarize_patterns(row),
        'Surge_Prob_%':       round(prob * 100, 1),
        'Above_Threshold':    'YES' if prob >= opt_thresh else 'NO',
        'Confidence':         confidence_band(prob, opt_thresh),
    })
    if i % 50 == 0 or i == total:
        print(f'  {i}/{total} done...')

all_picks = pd.DataFrame(results).sort_values('Surge_Prob_%', ascending=False).reset_index(drop=True)
all_picks.insert(0, 'Rank', all_picks.index + 1)
print(f'\nDone — {len(all_picks)} stocks ranked.')
print(f'Stocks ABOVE threshold ({opt_thresh:.2f}): {(all_picks["Above_Threshold"]=="YES").sum()}')

In [ ]:
# Show all stocks sorted high → low
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 35)

print(all_picks[[
    'Rank','Symbol','Latest_Return_%','RSI','Volume_Surge_x',
    'Corp_Ann_Sentiment','News_Sentiment','Blended_Sentiment',
    'Patterns_Detected','Surge_Prob_%','Above_Threshold','Confidence'
]].to_string(index=False))

out_path = os.path.join(ROOT, 'ai_ideal_stocks_report.csv')
all_picks.to_csv(out_path, index=False)
print(f'\nSaved → {out_path}')

## 10. Result Analysis & Charts

In [ ]:
# Surge probability distribution + confidence breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(all_picks['Surge_Prob_%'], bins=30, color='#58a6ff', edgecolor='black', alpha=0.8)
axes[0].axvline(opt_thresh*100, color='#facc15', lw=2, linestyle='--', label=f'Threshold {opt_thresh:.2f}')
axes[0].axvline((opt_thresh+0.20)*100, color='#4ade80', lw=2, linestyle='--', label='HIGH band')
axes[0].set_title('Surge Probability Distribution', fontweight='bold')
axes[0].set_xlabel('Surge Prob %'); axes[0].set_ylabel('Stock Count'); axes[0].legend()

conf_counts = all_picks['Confidence'].value_counts()
axes[1].pie(conf_counts.values, labels=conf_counts.index, autopct='%1.1f%%',
            colors=['#4ade80','#facc15','#94a3b8'][:len(conf_counts)], startangle=90)
axes[1].set_title('Confidence Band Distribution', fontweight='bold')

plt.tight_layout(); plt.show()

above = all_picks[all_picks['Above_Threshold']=='YES']
print(f'Above threshold  : {len(above)}')
print(f'HIGH confidence  : {(all_picks["Confidence"].str.startswith("HIGH")).sum()}')
print(f'MEDIUM confidence: {(all_picks["Confidence"].str.startswith("MEDIUM")).sum()}')
print(f'LOW confidence   : {(all_picks["Confidence"].str.startswith("LOW")).sum()}')

In [ ]:
# Top 20 horizontal bar chart
top20 = all_picks.head(20)
fig, ax = plt.subplots(figsize=(11, 7))
bar_colors = ['#4ade80' if c.startswith('HIGH') else '#facc15' if c.startswith('MED') else '#94a3b8'
              for c in top20['Confidence']]
bars = ax.barh(top20['Symbol'][::-1], top20['Surge_Prob_%'][::-1], color=bar_colors[::-1])
ax.set_xlabel('Surge Probability %')
ax.set_title(f'Top 20 AI Picks — High → Low (threshold={opt_thresh:.2f})', fontweight='bold')
ax.axvline(opt_thresh*100, color='#facc15', lw=1.5, linestyle='--')
for bar, val in zip(bars[::-1], top20['Surge_Prob_%']):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8)
high_p = mpatches.Patch(color='#4ade80', label='HIGH')
med_p  = mpatches.Patch(color='#facc15', label='MEDIUM')
low_p  = mpatches.Patch(color='#94a3b8', label='LOW')
ax.legend(handles=[high_p,med_p,low_p], fontsize=9)
ax.grid(axis='x', alpha=0.2); plt.tight_layout(); plt.show()

In [ ]:
# Pattern frequency: buy candidates vs all
buy_cands = all_picks[all_picks['Above_Threshold']=='YES']
pat_names = ['Breakout','Consolidation','Gap-Up','Accumulation','Engulfing','Hammer','Doji']
pat_buy = {p: buy_cands['Patterns_Detected'].str.contains(p, na=False).sum() for p in pat_names}
pat_all = {p: all_picks['Patterns_Detected'].str.contains(p, na=False).sum() for p in pat_names}

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(pat_names))
ax.bar([i-0.2 for i in x], pat_buy.values(), width=0.35, color='#4ade80', label='Buy Candidates')
ax.bar([i+0.2 for i in x], pat_all.values(),  width=0.35, color='#58a6ff', label='All stocks', alpha=0.7)
ax.set_xticks(list(x)); ax.set_xticklabels(pat_names, rotation=20, ha='right')
ax.set_title('Pattern Frequency: Buy Candidates vs All Stocks', fontweight='bold')
ax.set_ylabel('Count'); ax.legend(); ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

print(f'Buy candidates : {len(buy_cands)}')
print(f'Most common patterns in buy candidates:')
print(sorted(pat_buy.items(), key=lambda x: x[1], reverse=True))